# Feature Engineering

In this notebook new features will be created to better describe the songs, covering lyrical complexity, sonic characteristics, and contextual metadata. 

A correlation analysis will be performed to cut down columns with too similar values based on the pearson correlation matrix. The result of the new dataset can be found in the engineered_dataset.cv

In [1]:
import pandas as pd
import numpy as np
import altair as alt
from sklearn.preprocessing import MinMaxScaler
from os import path

dataset_path = path.join('..', 'dataset', 'cleaned_tracks.csv')
df = pd.read_csv(dataset_path, sep=',')

## Lyrical Features

### words_per_minute

The number of tokens (words) divided by the track duration in minutes.
Rap styles vary significantly by speed his feature distinguishes technical/fast flow rap from mumble Rap or melodic trap, which tends to be slower and more spaced out.

In [2]:
df['words_per_minute'] = df['n_tokens'] / (df['duration_ms'] / 60000)

### syllables_per_beat

The number of tokens (words) processed relative to the musical beat (BPM) rather than just time. Unlike Words Per Minute, this measures how "full" the flow is relative to the base of the song.

In [3]:
df['syllables_per_beat'] = df['n_tokens'] / ((df['duration_ms'] / 60000) * df['bpm'])

### explicitness_density

The ratio of total swear words (Italian + English) to the total number of tokens. Captures the "explicitness" of the content.

In [4]:
df['explicitness_density'] = (df['swear_IT'].fillna(0) + df['swear_EN'].fillna(0)) / df['n_tokens']
df['explicitness_density'] = df['explicitness_density'].fillna(0)  #handle div by zero

### lyrical_complexity

A composite metric combining vocabulary richness (lexical_density) with the sophistication of the words used (char_per_tok). Differentiates more "original" rap (high density, longer words) from repetitive trap (low density, shorter words).

In [5]:
df['lyrical_complexity'] = df['lexical_density'] * df['char_per_tok']

## Audio & Musical Features

### audio_aggressiveness

A composite score of audio features associated with noisy and intense sounds (Loudness, Rolloff, and Zero Crossing Rate). Distinguishes tracks with distorted basses and aggressive productions (e.g., Drill, Industrial) from smoother, cleaner productions (e.g., Lo-fi, Old School).

In [6]:
#scaler for composite features (requires normalization (Min-Max) of components first because they have different scales)
scaler = MinMaxScaler()

audio_feats_agg = ['rolloff', 'loudness', 'zcr']
norm_agg = df[audio_feats_agg].copy()
norm_agg = norm_agg.fillna(norm_agg.mean()) #simple imputation for the calculation
norm_agg_scaled = scaler.fit_transform(norm_agg)

df['audio_aggressiveness'] = norm_agg_scaled.mean(axis=1)

### harmonic_complexity

An indicator of how rich the instrumental is, combining spectral_complexity (number of peaks) and flatness (inverse of noisiness).
High complexity and low flatness usually indicate a rich melody with many instruments. Low complexity and high flatness suggest a minimalist, perhaps noisy or drum-focused beat.

In [7]:
df['harmonic_complexity'] = df['spectral_complexity'] * (1 - df['flatness'])

### vocal_clarity

An estimation of how "clear" or "bright" the vocal mix is, using rolloff relative to loudness. Modern productions often have very high rolloff (crisp vocals) and high loudness. Lo-fi or "lower" productions might have lower rolloff despite high loudness.

In [8]:
df['vocal_clarity'] = df['rolloff'] / df['loudness'].abs()

## Contextual & Metadata Features

### collab_count

The number of featured artists on a track.

In [9]:
def count_featured(val):
    if pd.isna(val):
        return 0
    return len(str(val).split(','))

df['collab_count'] = df['featured_artists'].apply(count_featured)

### artist_rel_popularity

The popularity of a song normalized by the artist's average popularity.

In [10]:
df['artist_mean_pop'] = df.groupby('name_artist')['popularity'].transform('mean')
df['artist_std_pop'] = df.groupby('name_artist')['popularity'].transform('std')

df['artist_rel_popularity'] = (df['popularity'] - df['artist_mean_pop']) / df['artist_std_pop']
df['artist_rel_popularity'] = df['artist_rel_popularity'].fillna(0) #handle artists with 1 song

### season_rel_popularity

The popularity of a track standardized (Z-score) against the average popularity of all tracks released in the same season (Winter, Spring, Summer, Autumn). A song released in summer might need a higher raw popularity score to be considered a standout "hit" compared to a song released in a quieter season.

In [11]:
def get_season(month):
    if pd.isna(month):
        return np.nan
    month = int(month)
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    return np.nan

df['release_season'] = df['month'].apply(get_season)

#Mean and Std Dev of popularity for each season
season_stats = df.groupby('release_season')['popularity'].agg(['mean', 'std']).reset_index()
season_stats.rename(columns={'mean': 'season_mean_pop', 'std': 'season_std_pop'}, inplace=True)

df = df.merge(season_stats, on='release_season', how='left')

#compute Z-score: (Popularity - Season Mean) / Season Std
df['season_rel_popularity'] = (df['popularity'] - df['season_mean_pop']) / df['season_std_pop']

df['season_rel_popularity'] = df['season_rel_popularity'].fillna(0)

print(df[['title', 'release_season', 'popularity', 'season_rel_popularity']].head())

         title release_season  popularity  season_rel_popularity
0  polka 2 :-/         Spring        46.0               0.691818
1        POLKA         Spring        46.0               0.691818
2  britney ;-)         Winter        39.0               0.402718
3          CEO         Spring        47.0               0.744580
4       LONDRA         Spring        41.0               0.428008


## Feature Correlation Analysis

We analyze the correlation between new features and old numerical columns to assess their potential correlation to old columns.

In [12]:
base_numerical_cols = [
    'popularity', 'stats_pageviews', 'duration_ms',
    'bpm', 'centroid', 'rolloff', 'flux', 'flatness', 'spectral_complexity', 'pitch', 'loudness',
    'n_sentences', 'n_tokens', 'tokens_per_sent', 'char_per_tok', 'lexical_density', 'avg_token_per_clause',
    'swear_IT', 'swear_EN'
]

new_features = [
    'words_per_minute', 'syllables_per_beat', 'explicitness_density', 
    'lyrical_complexity', 'audio_aggressiveness', 'harmonic_complexity',
    'vocal_clarity', 'collab_count', 'artist_rel_popularity', 'season_rel_popularity'
]

cols_to_use = base_numerical_cols + new_features

exclude_cols = ['zcr', 'rms']
cols_to_use = [col for col in cols_to_use if col not in exclude_cols and col in df.columns]

df_corr = df[cols_to_use].copy()
corr_matrix = df_corr.corr()

corr_long = corr_matrix.stack().reset_index()
corr_long.columns = ['Variable 1', 'Variable 2', 'Correlation']
corr_long['Correlation_Label'] = corr_long['Correlation'].apply(lambda x: f"{x:.2f}")

base = alt.Chart(corr_long).encode(
    x=alt.X('Variable 2', title=None, sort=cols_to_use),
    y=alt.Y('Variable 1', title=None, sort=cols_to_use)
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        'Correlation',
        scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
        legend=alt.Legend(title="Correlation")
    ),
    tooltip=['Variable 1', 'Variable 2', alt.Tooltip('Correlation', format='.2f')]
)

text = base.mark_text(size=8).encode(
    text='Correlation_Label',
    color=alt.condition(
        alt.expr.abs(alt.datum.Correlation) > 0.5,
        alt.value('white'),
        alt.value('black')
    )
)

chart = (heatmap + text).properties(
    title='Pearson Correlation Matrix (Audio, Lyrics, Metadata & Engineered Features)',
    width=950,
    height=950
)

chart.interactive()

alt.LayerChart(...)